# 07. Does the combined loss actually help?

Notebook 05 trains with `CoSENTLoss` and `CosineSimilarityLoss` together. The README explains
this as giving the model gradient signal for ranking and for absolute score at the same time.
Notebooks 01 to 04 all use CoSENT alone.

So the one architectural decision that separates the production model from the exploratory
study has never been tested. This notebook tests it.

## Pre-registered hypothesis

Written before the runs, so the prediction cannot be fitted to the outcome after the fact.

CoSENT is a rank-based objective. It pushes `cos(i) > cos(j)` whenever `label(i) > label(j)`
and is otherwise indifferent to scale, so it has no reason to make cosines match label
magnitudes. That predicts **arm A ranks well and calibrates badly**, which is exactly what
notebook 01 observed: fine-tuning roughly doubled Spearman while making MAE worse.

`CosineSimilarityLoss` is MSE between cosine and label. It targets magnitude directly, so
**arm B should calibrate better and rank worse**.

The sharper prediction concerns arm C. An oracle analysis on the current production model
(an isotonic calibrator fitted directly on the test set, which is a ceiling and not a
reportable metric) showed that after Platt calibration only **0.019 MAE** remained
recoverable by any monotone calibrator. If almost all the magnitude information is already
retrievable post hoc, the `CosineSimilarityLoss` term is doing work that calibration does
anyway, and:

> **H1.** Arm A + Platt will be statistically indistinguishable from arm C + Platt.

If H1 holds, the second loss term should be dropped and the model gets simpler at no cost.

## Outcomes, committed in advance

| Result | Action |
|---|---|
| C beats A | The README claim becomes evidence-backed rather than asserted |
| C ties A | Drop the second loss term, report that calibration subsumed it |
| A beats C | Ship the simpler model, write up why the combined objective hurt |

All three are reported. Selecting the arm that flatters the current design would make every
other number in this repository worth less.

## Experimental design

**One variable.** Same 815 training pairs, same 212 external pairs, same `split_seed=42`
calibration/test split, same augmentation, same 4 epochs, same batch size, same seeds
42/43/44, same base checkpoint. Only the loss changes.

**Gradient steps are matched across arms.** This is the part a naive ablation gets wrong.
`SentenceTransformer.fit()` performs one backward pass *per training objective per step*, so
passing two objectives doubles the gradient updates per epoch. Comparing a two-objective arm
against a one-objective arm would confound the loss with twice the optimisation, and a win
for the combined arm could be nothing more than a longer effective schedule.

Every arm therefore receives **two dataloaders**:

| Arm | Objective 1 | Objective 2 |
|---|---|---|
| `cosent` | CoSENT | CoSENT |
| `cosine` | CosineSimilarity | CosineSimilarity |
| `combined` | CoSENT | CosineSimilarity |

Identical step counts, identical data exposure, identical warmup schedule. The only
difference between arms is which loss is applied to the second pass.

**The split is deliberately not changed.** A posting-grouped calibration/test split is the
better design and is noted as future work, but changing it here would move the test set and
break comparability with notebook 05's published numbers. One variable at a time.

In [1]:
# ============================================================
# CELL 1: Config, dependencies, seeding
# ============================================================
!pip install -q sentence-transformers datasets pandas scikit-learn matplotlib

# Mirrors Notebooks/05_production.ipynb exactly. If these drift apart the ablation stops
# describing the production recipe, so they are asserted against the published config below.
CONFIG = {
    "seeds":               [42, 43, 44],
    "epochs":              4,
    "batch_size":          16,
    "n_augment":           3,
    "max_jd_words":        350,
    "split_seed":          42,
    "val_fraction":        0.15,
    "bootstrap_resamples": 2000,
}

ARMS = ["cosent", "cosine", "combined"]

import os, re, json, time, random, pickle, warnings
import numpy as np
import pandas as pd
import torch
warnings.filterwarnings('ignore')

from collections import Counter
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.isotonic import IsotonicRegression
from scipy.stats import spearmanr
from scipy.optimize import minimize

from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator


def set_seed(seed):
    """Seed every RNG that can affect a training run. Call before each .fit()."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# use_amp defeats the seeding above: GradScaler adapts its loss scale in response to observed
# overflows, so two runs of an identical config diverge. It is kept ON here because notebook
# 05 uses it and this ablation must describe the production recipe, not an idealised one.
# Consequence: arms are comparable to each other within this session, and re-running the
# notebook will not reproduce these numbers bit for bit. That is why every arm reports 3
# seeds with a spread rather than a single figure.
USE_AMP = True

print(f"PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Arms: {ARMS} x seeds {CONFIG['seeds']} = {len(ARMS)*len(CONFIG['seeds'])} runs")
print(f"Estimated: ~{len(ARMS)*len(CONFIG['seeds'])*24/60:.1f} hours on a T4")

PyTorch 2.11.0+cu128 | CUDA True
GPU: Tesla T4
Arms: ['cosent', 'cosine', 'combined'] x seeds [42, 43, 44] = 9 runs
Estimated: ~3.6 hours on a T4


In [2]:
# ============================================================
# CELL 2: Load, verify no JD leakage, split
# ============================================================
from google.colab import files

print("Upload TRAINING data (resume_jd_training_800.csv):")
df = pd.read_csv(list(files.upload().keys())[0])
print(f"  Training: {len(df)} pairs | {df['jd'].nunique()} unique JDs")

print("\nUpload EXTERNAL TEST data (external_test_200_pairs.csv):")
ext_full = pd.read_csv(list(files.upload().keys())[0])
print(f"  External: {len(ext_full)} pairs | {ext_full['jd'].nunique()} unique JDs")

# The entire external-validation argument rests on this.
overlap = set(df['jd']) & set(ext_full['jd'])
assert len(overlap) == 0, f"LEAKAGE: {len(overlap)} shared JDs, external results are invalid"
print(f"\nJD overlap between training and external: 0, verified")

ext_cal, ext_test = train_test_split(
    ext_full, test_size=0.5, random_state=CONFIG["split_seed"], stratify=ext_full["match_type"])
ext_cal, ext_test = ext_cal.copy(), ext_test.copy()

print(f"External calibration: {len(ext_cal)} | External final test: {len(ext_test)}")
print(f"Final test distribution: {dict(Counter(ext_test['match_type']))}")

# ── The comparison is only paired if this is notebook 05's test set ──
# Every claim here is "arm X beats arm Y on the same pairs", and the results are also
# compared against notebook 05's published production numbers. A different split would still
# produce entirely plausible figures while quietly answering a different question, which is
# the failure mode this repository has already been bitten by twice.
import hashlib
fingerprint = hashlib.md5(
    ','.join(map(str, sorted(ext_test['id'].tolist()))).encode()).hexdigest()[:12]
EXPECTED = "f7e6ee8fcbb3"   # from Results/demo_pairs.json, the split notebook 05 shipped
print(f"\nTest-set fingerprint: {fingerprint}")
assert fingerprint == EXPECTED, (
    f"Split mismatch: got {fingerprint}, expected {EXPECTED}. This is not the 106 pairs "
    f"notebook 05 reported on, so no comparison in this notebook is paired. Check that the "
    f"uploaded CSV is Data/external_test_200_pairs.csv and split_seed is 42.")
print("  verified: identical to the final test split notebook 05 published")

Upload TRAINING data (resume_jd_training_800.csv):


Saving resume_jd_training_800.csv to resume_jd_training_800.csv
  Training: 815 pairs | 255 unique JDs

Upload EXTERNAL TEST data (external_test_200_pairs.csv):


Saving external_test_200_pairs.csv to external_test_200_pairs.csv
  External: 212 pairs | 53 unique JDs

JD overlap between training and external: 0, verified
External calibration: 106 | External final test: 106
Final test distribution: {'partial': 13, 'strong': 27, 'good': 14, 'weak': 26, 'hard_negative': 26}

Test-set fingerprint: f7e6ee8fcbb3
  verified: identical to the final test split notebook 05 published


In [3]:
# ============================================================
# CELL 3: Smart truncation + augmentation  (verbatim from notebook 05)
# ============================================================
def smart_truncate_jd(jd_text, max_words=CONFIG["max_jd_words"]):
    boilerplate = [r'equal opportunity employer.*', r'does not discriminate.*',
                   r'reasonable accommodation.*', r'(compensation|salary|pay) range.*',
                   r'(what we offer|our benefits|perks and benefits|benefits:).*']
    cleaned = jd_text
    for p in boilerplate:
        cleaned = re.split(p, cleaned, flags=re.IGNORECASE)[0]
    req = [r'(requirements?|qualifications?|what you.?ll need|must have)',
           r'(responsibilities|what you.?ll do|in this role|you will)']
    best = 0
    for p in req:
        m = re.search(p, cleaned, re.IGNORECASE)
        if m:
            s = max(0, m.start() - 200)
            if best == 0 or s < best:
                best = s
    if best > 100:
        cleaned = cleaned[:100] + "\n" + cleaned[best:]
    words = cleaned.split()
    return ' '.join(words[:max_words]).strip() if len(words) > max_words else cleaned.strip()


for d in (df, ext_cal, ext_test, ext_full):
    d['jd_clean'] = d['jd'].apply(smart_truncate_jd)


def shuffle_sections(text):
    sections, curr, lines = {}, 'header', []
    for line in text.split('\n'):
        u = line.strip().upper()
        for kw, sec in [('SUMMARY','summary'), ('SKILLS','skills'),
                        ('EXPERIENCE','experience'), ('EDUCATION','education')]:
            if kw in u:
                sections[curr] = '\n'.join(lines); curr = sec; lines = [line]; break
        else:
            lines.append(line)
    sections[curr] = '\n'.join(lines)
    body = ['summary', 'skills', 'experience', 'education']
    random.shuffle(body)
    return (sections.get('header','') + ''.join(sections.get(s,'') for s in body if s in sections)).strip()


def drop_sentences(text, rate=0.12):
    sents = re.split(r'(?<=[.!?])\s+|\n', text)
    if len(sents) <= 3:
        return text
    drop = set(random.sample(range(1, len(sents)),
                             min(max(1, int(len(sents)*rate)), len(sents)-2)))
    return ' '.join(s for i, s in enumerate(sents) if i not in drop).strip()


def keyword_noise(text):
    reps = {'Python':'Python programming', 'JavaScript':'JS/JavaScript', 'React':'React.js',
            'AWS':'Amazon Web Services', 'SQL':'SQL databases', 'Docker':'Docker containers',
            '5+ years':'five or more years', '3+ years':'three or more years'}
    out = text
    for k in random.sample(list(reps), min(2, len(reps))):
        if k in out:
            out = out.replace(k, reps[k], 1)
    return out


def augment_dataset(frame, n_aug=CONFIG["n_augment"]):
    rows, kinds = [], ['shuffle', 'drop_jd', 'drop_resume', 'noise']
    for _, row in frame.iterrows():
        rows.append({'resume': row['resume'], 'jd': row['jd_clean'],
                     'score': row['score'], 'match_type': row['match_type'], 'augmented': False})
        for kind in random.sample(kinds, min(n_aug, len(kinds))):
            r, j = row['resume'], row['jd_clean']
            if   kind == 'shuffle':     r = shuffle_sections(r)
            elif kind == 'drop_jd':     j = drop_sentences(j)
            elif kind == 'drop_resume': r = drop_sentences(r)
            else:                       r = keyword_noise(r)
            rows.append({'resume': r, 'jd': j,
                         'score': float(np.clip(row['score'] + np.random.uniform(-0.02, 0.02), 0, 1)),
                         'match_type': row['match_type'], 'augmented': True})
    return pd.DataFrame(rows)


print("Preprocessing and augmentation ready")

Preprocessing and augmentation ready


In [4]:
# ============================================================
# CELL 4: Metrics, calibrators  (verbatim from notebook 05)
# ============================================================
class PlattCalibrator:
    """2-parameter sigmoid: calibrated = 1/(1+exp(-(a*raw+b)))."""
    def __init__(self):
        self.a, self.b = 1.0, 0.0

    def fit(self, raw_scores, true_scores):
        raw, true = np.asarray(raw_scores), np.asarray(true_scores)
        def loss(params):
            a, b = params
            return float(np.mean((1.0/(1.0+np.exp(-(a*raw+b))) - true)**2))
        self.a, self.b = minimize(loss, x0=[1.0, 0.0], method='Nelder-Mead').x
        return self

    def __call__(self, scores):
        s = np.asarray(scores)
        return (1.0/(1.0+np.exp(-(self.a*s+self.b)))).tolist()


def raw_scores(model, frame):
    """Row-wise cosine between resume and cleaned-JD embeddings."""
    r = model.encode(frame['resume'].tolist(), show_progress_bar=False, convert_to_numpy=True)
    j = model.encode(frame['jd_clean'].tolist(), show_progress_bar=False, convert_to_numpy=True)
    r = r / np.linalg.norm(r, axis=1, keepdims=True)
    j = j / np.linalg.norm(j, axis=1, keepdims=True)
    return (r*j).sum(axis=1).tolist()


def metrics(true, pred):
    sp, _ = spearmanr(true, pred)
    return {"spearman": round(float(sp), 4),
            "mae": round(float(np.mean(np.abs(np.asarray(true)-np.asarray(pred)))), 4)}


test_true = ext_test['score'].astype(float).tolist()
cal_true  = ext_cal['score'].astype(float).tolist()

train_df, val_df = train_test_split(
    df, test_size=CONFIG["val_fraction"], random_state=CONFIG["split_seed"], stratify=df["match_type"])
train_df, val_df = train_df.copy(), val_df.copy()
print(f"Train {len(train_df)} | Val {len(val_df)} | Cal {len(ext_cal)} | Test {len(ext_test)}")

Train 692 | Val 123 | Cal 106 | Test 106


In [5]:
# ============================================================
# CELL 5: The ablation. 3 arms x 3 seeds, step-matched
# ============================================================
def objectives_for(arm, model, dl_a, dl_b):
    """Two objectives for every arm, so gradient-step count is identical across arms.

    Passing a single objective would halve the updates per epoch for the single-loss arms and
    confound 'the combined loss helps' with 'twice the optimisation helps'.
    """
    cosent = lambda: losses.CoSENTLoss(model=model)
    cosine = lambda: losses.CosineSimilarityLoss(model=model)
    if arm == "cosent":
        return [(dl_a, cosent()), (dl_b, cosent())]
    if arm == "cosine":
        return [(dl_a, cosine()), (dl_b, cosine())]
    if arm == "combined":
        return [(dl_a, cosent()), (dl_b, cosine())]
    raise ValueError(arm)


runs = []
PARTIAL = "loss_ablation_partial.json"

# Colab disconnects. Every completed run is written to disk immediately so a dropped session
# costs one run rather than the whole experiment; re-running this cell skips what is done.
if os.path.exists(PARTIAL):
    runs = json.load(open(PARTIAL))
    print(f"Resuming: {len(runs)} run(s) already complete\n")

done = {(r["arm"], r["seed"]) for r in runs}

for arm in ARMS:
    for seed in CONFIG["seeds"]:
        if (arm, seed) in done:
            print(f"skip {arm} seed {seed} (already done)")
            continue
        print("=" * 70)
        print(f"ARM {arm}  |  SEED {seed}")
        print("=" * 70)
        set_seed(seed)

        aug_df = augment_dataset(train_df)
        examples = [InputExample(texts=[r['resume'], r['jd']], label=float(r['score']))
                    for _, r in aug_df.iterrows()]
        dl_a = DataLoader(examples, shuffle=True, batch_size=CONFIG["batch_size"])
        dl_b = DataLoader(examples, shuffle=True, batch_size=CONFIG["batch_size"])
        evaluator = EmbeddingSimilarityEvaluator(
            sentences1=val_df['resume'].tolist(),
            sentences2=val_df['jd_clean'].tolist(),
            scores=val_df['score'].astype(float).tolist(), name='val')

        model = SentenceTransformer('all-mpnet-base-v2')
        out_dir = f"models/{arm}_seed{seed}"

        set_seed(seed)   # re-seed immediately before fit
        t0 = time.time()
        model.fit(
            train_objectives=objectives_for(arm, model, dl_a, dl_b),
            evaluator=evaluator, epochs=CONFIG["epochs"],
            warmup_steps=int(len(dl_a)*0.1), evaluation_steps=len(dl_a),
            output_path=out_dir, show_progress_bar=True, use_amp=USE_AMP)
        mins = (time.time()-t0)/60

        # Same defect guard as notebook 05: fit() saves the best-by-validation checkpoint
        # while the in-memory model is the final epoch. Measure what was saved.
        del model
        torch.cuda.empty_cache()
        model = SentenceTransformer(out_dir)

        cal_raw = raw_scores(model, ext_cal)
        t_raw   = raw_scores(model, ext_test)
        platt   = PlattCalibrator().fit(cal_raw, cal_true)
        iso     = IsotonicRegression(out_of_bounds='clip').fit(cal_raw, cal_true)

        run = {"arm": arm, "seed": seed, "train_minutes": round(mins, 1),
               "raw":      metrics(test_true, t_raw),
               "platt":    metrics(test_true, platt(t_raw)),
               "isotonic": metrics(test_true, iso.predict(t_raw).tolist()),
               "test_raw": [round(float(v), 6) for v in t_raw],
               "cal_raw":  [round(float(v), 6) for v in cal_raw],
               "model_path": out_dir}
        runs.append(run)
        json.dump(runs, open(PARTIAL, "w"))

        print(f"  [{mins:.0f} min]  raw {run['raw']} | platt {run['platt']} | iso {run['isotonic']}")

        del model
        torch.cuda.empty_cache()

print(f"\nComplete: {len(runs)} runs")

ARM cosent  |  SEED 42


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  [25 min]  raw {'spearman': 0.8381, 'mae': 0.1468} | platt {'spearman': 0.8381, 'mae': 0.114} | iso {'spearman': 0.8322, 'mae': 0.1144}

Complete: 9 runs


In [6]:
# ============================================================
# CELL 6: Aggregate per arm
# ============================================================
def agg(arm, key, metric):
    v = [r[key][metric] for r in runs if r["arm"] == arm]
    return float(np.mean(v)), float(np.std(v))


aggregate = {}
print(f"{'Arm':<10} {'variant':<10} {'Spearman':>20} {'MAE':>20}")
print("-" * 62)
for arm in ARMS:
    aggregate[arm] = {}
    for key in ("raw", "platt", "isotonic"):
        sm, ss = agg(arm, key, "spearman"); mm, ms = agg(arm, key, "mae")
        aggregate[arm][key] = {"spearman_mean": round(sm, 4), "spearman_std": round(ss, 4),
                               "mae_mean": round(mm, 4), "mae_std": round(ms, 4)}
        print(f"{arm:<10} {key:<10} {sm:>12.4f} +/- {ss:.4f} {mm:>12.4f} +/- {ms:.4f}")
    print()

print("Per-seed Platt Spearman (every run, including the bad ones):")
for arm in ARMS:
    vals = [(r["seed"], r["platt"]["spearman"]) for r in runs if r["arm"] == arm]
    print(f"  {arm:<10} " + "  ".join(f"seed {s}: {v:.4f}" for s, v in sorted(vals)))

Arm        variant                Spearman                  MAE
--------------------------------------------------------------
cosent     raw              0.8477 +/- 0.0239       0.2199 +/- 0.0229
cosent     platt            0.8477 +/- 0.0239       0.1078 +/- 0.0078
cosent     isotonic         0.8440 +/- 0.0263       0.1088 +/- 0.0056

cosine     raw              0.8045 +/- 0.0092       0.1486 +/- 0.0155
cosine     platt            0.8045 +/- 0.0092       0.1261 +/- 0.0090
cosine     isotonic         0.8147 +/- 0.0153       0.1236 +/- 0.0076

combined   raw              0.8447 +/- 0.0143       0.1460 +/- 0.0097
combined   platt            0.8447 +/- 0.0143       0.1110 +/- 0.0079
combined   isotonic         0.8337 +/- 0.0188       0.1093 +/- 0.0079

Per-seed Platt Spearman (every run, including the bad ones):
  cosent     seed 42: 0.8810  seed 43: 0.8263  seed 44: 0.8357
  cosine     seed 42: 0.8145  seed 43: 0.7923  seed 44: 0.8067
  combined   seed 42: 0.8645  seed 43: 0.8314  seed 4

### Reading the table above

The pre-registered prediction was that `cosent` ranks well and calibrates badly, `cosine`
does the reverse, and `combined` after Platt lands on top of `cosent` after Platt because
calibration already recovers the magnitude information.

The `raw` rows are where the loss shows itself most directly, since no calibrator has run
yet. The `platt` rows are what the production model actually ships, and are the rows H1 is
about.

Point estimates alone do not settle this. Cell 8 exports per-pair predictions and the paired
cluster bootstrap runs locally, against the same 106 pairs, using the implementation in
`scripts/significance.py` rather than a second copy written here.

In [7]:
# ============================================================
# CELL 7: Seed ensembling, free with the runs already paid for
# ============================================================
# Three models per arm were trained and two were about to be discarded. Averaging the raw
# cosines of all three uses every run, so there is no selection step and nothing to cherry
# pick. This is the honest use of multiple seeds; picking the best one is not.
ensemble = {}
print(f"{'Arm':<10} {'best single':>14} {'median single':>15} {'3-seed ensemble':>17}")
print("-" * 58)
for arm in ARMS:
    arm_runs = [r for r in runs if r["arm"] == arm]
    if len(arm_runs) < 2:
        continue
    cal_stack  = np.mean([r["cal_raw"]  for r in arm_runs], axis=0)
    test_stack = np.mean([r["test_raw"] for r in arm_runs], axis=0)

    platt = PlattCalibrator().fit(cal_stack, cal_true)
    ens = {"raw": metrics(test_true, test_stack),
           "platt": metrics(test_true, platt(test_stack)),
           "n_models": len(arm_runs),
           "test_raw": [round(float(v), 6) for v in test_stack]}
    ensemble[arm] = ens

    singles = sorted(r["platt"]["spearman"] for r in arm_runs)
    print(f"{arm:<10} {singles[-1]:>14.4f} {singles[len(singles)//2]:>15.4f} "
          f"{ens['platt']['spearman']:>17.4f}")

print("\nEnsembling reduces variance by construction. Whether it also raises the mean is")
print("what the numbers above answer, and it is tested properly in the local bootstrap.")

Arm           best single   median single   3-seed ensemble
----------------------------------------------------------
cosent             0.8810          0.8357            0.8639
cosine             0.8145          0.8067            0.8142
combined           0.8645          0.8381            0.8550

Ensembling reduces variance by construction. Whether it also raises the mean is
what the numbers above answer, and it is tested properly in the local bootstrap.


In [8]:
# ============================================================
# CELL 8: Export for the paired significance test
# ============================================================
# The statistics live in scripts/significance.py so there is exactly one implementation of
# the cluster bootstrap in this project. This cell only exports predictions.
median_of = {}
for arm in ARMS:
    arm_runs = [r for r in runs if r["arm"] == arm]
    maes = [r["platt"]["mae"] for r in arm_runs]
    median_of[arm] = arm_runs[int(np.argsort(maes)[len(maes)//2])]

payload = {
    "_what": "Per-pair predictions for every arm on the identical 106-pair final test.",
    "_design": ("Three losses, three seeds each, gradient-step matched by giving every arm "
                "two training objectives. Only the loss differs."),
    "config": CONFIG,
    "arms": ARMS,
    "runs": [{k: v for k, v in r.items() if k not in ("test_raw", "cal_raw")} for r in runs],
    "aggregate": aggregate,
    "ensemble": {a: {k: v for k, v in e.items() if k != "test_raw"} for a, e in ensemble.items()},
    "pairs": [],
}

for i, (_, row) in enumerate(ext_test.reset_index(drop=True).iterrows()):
    preds = {}
    for arm in ARMS:
        med = median_of[arm]
        platt = PlattCalibrator().fit(med["cal_raw"], cal_true)
        preds[f"{arm}_raw"] = round(float(med["test_raw"][i]), 4)
        preds[f"{arm}_platt"] = round(float(platt(med["test_raw"])[i]), 4)
        if arm in ensemble:
            preds[f"{arm}_ensemble_raw"] = round(float(ensemble[arm]["test_raw"][i]), 4)
    payload["pairs"].append({
        "id": int(row["id"]),
        "true": round(float(row["score"]), 4),
        "jd": row["jd"],                       # cluster key for the posting-level bootstrap
        "match_type": row["match_type"],
        "preds": preds,
    })

with open("loss_ablation.json", "w") as f:
    json.dump(payload, f, indent=1)

print(f"Wrote loss_ablation.json: {len(payload['pairs'])} pairs x {len(ARMS)} arms")
print(f"Median seed per arm: " + ", ".join(f"{a}={median_of[a]['seed']}" for a in ARMS))

from google.colab import files
files.download("loss_ablation.json")

Wrote loss_ablation.json: 106 pairs x 3 arms
Median seed per arm: cosent=44, cosine=43, combined=44


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Next step

Put `loss_ablation.json` in `Results/`, then run:

```bash
python scripts/significance.py --ablation
```

That runs the paired cluster bootstrap over postings, comparing every arm against
`combined`, and writes the verdict into `Results/loss_ablation_significance.json`.

Whatever it says goes into the README, including the outcome where the combined loss turns
out to have been unnecessary all along.